# Vision-Based Graph RAG with OraclePageIndex

<p align="center"><b>No OCR Needed &middot; Multimodal Ollama &middot; Oracle Property Graphs &middot; Page Image Reasoning</b></p>

---

## Introduction

Traditional document QA extracts text via OCR, then feeds it to an LLM. But if a
**vision-language model (VLM)** can already understand images, do we still need OCR?

This notebook demonstrates a **vision-based, vectorless RAG pipeline**:
1. Build a document tree with OraclePageIndex (reasoning-based structure extraction)
2. LLM searches the tree to find relevant sections
3. Extract the corresponding **PDF page images** (not text)
4. Send page images to a **multimodal Ollama model** (gemma3) to generate answers

The result: document QA that reasons directly over page images — no OCR step needed.

### What you'll learn:
- [x] Extract PDF pages as images with PyMuPDF
- [x] Reasoning-based tree search for relevant page identification
- [x] Vision-based answer generation with multimodal Ollama
- [x] Compare text-based vs. vision-based answers

> **Prerequisites:**
> - [Ollama](https://ollama.com/) running with a **vision-capable model** (`ollama pull mistral:7b` for text, `ollama pull gemma3` for vision)

---

## Step 0: Preparation

In [1]:
import sys, os, json, base64, textwrap
sys.path.insert(0, os.path.abspath(".."))

import fitz  # PyMuPDF

from oracle_pageindex.llm import OllamaClient
from oracle_pageindex.parser import DocumentParser

### Configure models

In [2]:
# Text model for tree parsing and search
TEXT_MODEL = "mistral:7b"

# Vision model for multimodal image reasoning
# gemma3, llava, llama3.2-vision all support vision
VLM_MODEL = "gemma3"

llm = OllamaClient(
    base_url="http://localhost:11434",
    model=TEXT_MODEL,
    temperature=0,
    num_ctx=8192,
)

response = llm.chat("Say 'ready' in one word.")
print(f"Text LLM ({TEXT_MODEL}) ready: {response}")
print(f"Vision model: {VLM_MODEL}")

Text LLM (mistral:7b) ready:  Ready
Vision model: gemma3


### PDF page image extraction helpers

In [3]:
def extract_pdf_page_images(pdf_path, output_dir="pdf_images"):
    """Convert each PDF page to a JPEG image."""
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    page_images = {}
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        mat = fitz.Matrix(2.0, 2.0)  # 2x zoom for quality
        pix = page.get_pixmap(matrix=mat)
        img_path = os.path.join(output_dir, f"page_{page_num + 1}.jpg")
        with open(img_path, "wb") as f:
            f.write(pix.tobytes("jpeg"))
        page_images[page_num + 1] = img_path
    doc.close()
    print(f"Extracted {len(page_images)} page images to {output_dir}/")
    return page_images


def call_vlm(prompt, image_paths=None, model=None, max_retries=15):
    """Call Ollama with text + optional images (multimodal), with retry."""
    import httpx, time

    model = model or VLM_MODEL
    messages = [{"role": "user", "content": prompt}]

    # If images provided, use Ollama's multimodal format
    if image_paths:
        images_b64 = []
        for path in image_paths:
            with open(path, "rb") as f:
                images_b64.append(base64.b64encode(f.read()).decode("utf-8"))
        messages = [{
            "role": "user",
            "content": prompt,
            "images": images_b64,
        }]

    for attempt in range(max_retries):
        try:
            with httpx.Client(timeout=300.0) as client:
                resp = client.post(
                    f"{llm.base_url}/api/chat",
                    json={"model": model, "messages": messages, "stream": False,
                          "options": {"temperature": 0, "num_ctx": 8192}},
                )
                resp.raise_for_status()
                return resp.json()["message"]["content"]
        except Exception as e:
            wait = min(2 ** (attempt + 1), 60)
            print(f"  Ollama error (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  Max retries reached!")
                raise

## Step 1: Parse PDF and Extract Page Images

In [4]:
from urllib.request import urlretrieve

# Download "Attention Is All You Need" (15 pages, has figures/diagrams)
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
pdf_path = os.path.join("data", "attention-is-all-you-need.pdf")
os.makedirs("data", exist_ok=True)

if not os.path.exists(pdf_path):
    print(f"Downloading {pdf_url}...")
    urlretrieve(pdf_url, pdf_path)
    print(f"Saved to {pdf_path}")
else:
    print(f"Using cached {pdf_path}")

# Extract page images
print("\nExtracting page images...")
page_images = extract_pdf_page_images(pdf_path)

Using cached data/attention-is-all-you-need.pdf

Extracting page images...


Extracted 15 page images to pdf_images/


### Build the document tree

In [5]:
parser = DocumentParser(
    llm=llm,
    toc_check_page_num=20,
    max_token_num_each_node=20000,
    add_node_id=True,
    add_summaries=True,
)

print("Building document tree with Ollama...")
result = parser.build_tree(pdf_path)

tree = result["structure"]
page_list = result["page_list"]
total_pages = len(page_list)

print(f"Document: {result['doc_name']}")
print(f"Pages: {total_pages}")

Building document tree with Ollama...


Failed to parse JSON from LLM response


Failed to parse JSON from response


Could not parse ToC from LLM response


ToC extraction failed — using fallback structure


Document: attention-is-all-you-need.pdf
Pages: 15


### Visualize the tree

In [6]:
def print_tree(node, indent=0):
    if isinstance(node, list):
        for item in node:
            print_tree(item, indent)
        return
    title = node.get('title', 'Untitled')
    node_id = node.get('node_id', '?')
    start = node.get('start_index', '?')
    end = node.get('end_index', '?')
    prefix = '  ' * indent
    print(f"{prefix}[{node_id}] {title} (pages {start}-{end})")
    for child in node.get('nodes', []):
        print_tree(child, indent + 1)

def create_node_map(node, result=None):
    if result is None:
        result = {}
    if isinstance(node, list):
        for item in node:
            create_node_map(item, result)
        return result
    if isinstance(node, dict):
        nid = node.get('node_id')
        if nid:
            result[nid] = node
        for child in node.get('nodes', []):
            create_node_map(child, result)
    return result

def strip_text(node):
    if isinstance(node, list):
        return [strip_text(item) for item in node]
    if isinstance(node, dict):
        return {k: (strip_text(v) if k == 'nodes' else v)
                for k, v in node.items() if k != 'text'}
    return node

print("Document Tree:\n")
print_tree(tree)

Document Tree:

[0000] Page 1 (pages 1-2)
[0001] Page 2 (pages 2-3)
[0002] Page 3 (pages 3-4)
[0003] Page 4 (pages 4-5)
[0004] Page 5 (pages 5-6)
[0005] Page 6 (pages 6-7)
[0006] Page 7 (pages 7-8)
[0007] Page 8 (pages 8-9)
[0008] Page 9 (pages 9-10)
[0009] Page 10 (pages 10-11)
[0010] Page 11 (pages 11-12)
[0011] Page 12 (pages 12-13)
[0012] Page 13 (pages 13-14)
[0013] Page 14 (pages 14-15)
[0014] Page 15 (pages 15-15)


## Step 2: Reasoning-Based Tree Search

We send the tree structure (without raw text) to Ollama and ask it to reason about
which nodes contain the answer. This identifies the **page ranges** we need.

In [7]:
query = "What is the last operation in the Scaled Dot-Product Attention figure?"

tree_for_llm = strip_text(tree)

search_prompt = (
    "You are given a question and a tree structure of a document.\n"
    "Each node has a node_id, title, summary, and page range (start_index to end_index).\n"
    "Find all nodes that are likely to contain the answer.\n\n"
    f"Question: {query}\n\n"
    f"Document tree:\n{json.dumps(tree_for_llm, indent=2)}\n\n"
    'Reply in JSON format: '
    '{"thinking": "<reasoning>", "node_list": ["node_id_1", "node_id_2"]}\n'
    "Return ONLY valid JSON."
)

print(f"Question: {query}\n")
print("Searching tree with Ollama...\n")
search_result = llm.chat(search_prompt)
result_json = llm.extract_json(search_result)

print("Reasoning:")
print(textwrap.fill(result_json.get('thinking', ''), width=90))

print("\nRetrieved Nodes:")
node_map = create_node_map(tree)
for nid in result_json.get('node_list', []):
    node = node_map.get(nid, {})
    start = node.get('start_index', '?')
    end = node.get('end_index', '?')
    print(f"  [{nid}] {node.get('title', '?')} (pages {start}-{end})")

Question: What is the last operation in the Scaled Dot-Product Attention figure?

Searching tree with Ollama...



Reasoning:
<reasoning>

Retrieved Nodes:
  [0003] Page 4 (pages 4-5)
  [0004] Page 5 (pages 5-6)


## Step 3: Extract Page Images for Retrieved Nodes

Instead of extracting text, we get the **actual page images** from the PDF.
These will be sent directly to the VLM for visual reasoning.

In [8]:
# Collect page images for all retrieved nodes
retrieved_images = []
seen_pages = set()

for nid in result_json.get('node_list', []):
    node = node_map.get(nid, {})
    start = node.get('start_index', 1)
    end = node.get('end_index', start)
    for page_num in range(start, end + 1):
        if page_num not in seen_pages and page_num in page_images:
            retrieved_images.append(page_images[page_num])
            seen_pages.add(page_num)

print(f"Retrieved {len(retrieved_images)} page image(s) for visual context:")
for img in retrieved_images:
    print(f"  {img}")

Retrieved 3 page image(s) for visual context:
  pdf_images/page_4.jpg
  pdf_images/page_5.jpg
  pdf_images/page_6.jpg


## Step 4: Vision-Based Answer Generation

Now we send the page images directly to the multimodal Ollama model.
The VLM sees the actual PDF pages — figures, tables, formatting, and all —
and reasons over the visual content to answer the question.

In [9]:
answer_prompt = (
    "Answer the question based on the images of the document pages provided.\n\n"
    f"Question: {query}\n\n"
    "Provide a clear, concise answer based only on what you can see in the images."
)

print(f"Question: {query}\n")
print(f"Sending {len(retrieved_images)} page image(s) to {VLM_MODEL}...\n")

answer = call_vlm(answer_prompt, retrieved_images)

print("Answer (from visual reasoning):\n")
print(textwrap.fill(answer, width=90))

Question: What is the last operation in the Scaled Dot-Product Attention figure?

Sending 3 page image(s) to gemma3...



Answer (from visual reasoning):

SoftMax


## Step 5: Compare — Text vs. Vision

For comparison, let's answer the same question using text extracted from the same pages.

In [10]:
# Text-based answer (traditional approach)
text_parts = []
for nid in result_json.get('node_list', []):
    node = node_map.get(nid, {})
    text = node.get('text', '')
    if text:
        text_parts.append(text)

text_context = "\n\n".join(text_parts)

text_prompt = (
    "Answer the question based on the provided document text.\n\n"
    f"Question: {query}\n\n"
    f"Context:\n{text_context}\n\n"
    "Provide a clear, concise answer."
)

text_answer = llm.chat(text_prompt)

print("Answer (from text):\n")
print(textwrap.fill(text_answer, width=90))

print("\n" + "=" * 60)
print("\nThe vision-based answer can reference figures and diagrams")
print("that are lost during text extraction — this is the key advantage.")

Answer (from text):

 The last operation in the Scaled Dot-Product Attention figure is the multiplication of
the output of the softmax function with the values matrix (V).


The vision-based answer can reference figures and diagrams
that are lost during text extraction — this is the key advantage.


---

## Summary

This notebook demonstrated **vision-based RAG** with OraclePageIndex:

1. **Tree search** identifies relevant pages via LLM reasoning (no vectors)
2. **Page images** are extracted from the PDF (no OCR)
3. **Multimodal Ollama** (gemma3) reasons directly over the visual content

This approach is especially powerful for documents with:
- Complex figures and diagrams
- Tables with precise formatting
- Mathematical formulas
- Charts and graphs

---

*Built with [OraclePageIndex](https://github.com/jasperan/OraclePageIndex) — Oracle AI Database powered document intelligence with Property Graphs.*